<a href="https://www.kaggle.com/code/mrrogueknight/vandermonde-solver?scriptVersionId=335706240" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

---

# Vandermonde Polynomial Solver
## Theory & Implementation Guide

---

## 1. What is Polynomial Interpolation?

**Polynomial interpolation** is finding a polynomial that passes through a given set of points.

### The Problem
Given points: `(T₁, Y₁), (T₂, Y₂), ..., (Tₙ, Yₙ)`

Find a polynomial of degree `n-1` that passes through all points:

```
Y = aₙTⁿ + aₙ₋₁Tⁿ⁻¹ + ... + a₁T + a₀
```

### Example
Given 4 points:
```
(30.75, 867.2295), (30.88, 888.5687), (31.00, 875.7651), (31.12, 885.1544)
```

We find a cubic polynomial:
```
Y = a₃T³ + a₂T² + a₁T + a₀
```

---

## 2. The Vandermonde Matrix Method

### Step 1: Write Equations
For each point, substitute T and Y:

```
Point 1: a₃(30.75)³ + a₂(30.75)² + a₁(30.75) + a₀ = 867.2295
Point 2: a₃(30.88)³ + a₂(30.88)² + a₁(30.88) + a₀ = 888.5687
Point 3: a₃(31.00)³ + a₂(31.00)² + a₁(31.00) + a₀ = 875.7651
Point 4: a₃(31.12)³ + a₂(31.12)² + a₁(31.12) + a₀ = 885.1544
```

### Step 2: Matrix Form
This becomes a matrix equation:

```
[30.75³  30.75²  30.75  1]   [a₃]   [867.2295]
[30.88³  30.88²  30.88  1] × [a₂] = [888.5687]
[31.00³  31.00²  31.00  1]   [a₁]   [875.7651]
[31.12³  31.12²  31.12  1]   [a₀]   [885.1544]
```

The matrix is called the **Vandermonde Matrix**.

### Step 3: Solve
We solve the system using `numpy.linalg.solve()` to find `a₃, a₂, a₁, a₀`.

---

## 3. The Numerical Stability Problem

### Why is it a Problem?
When T values are **large and close together** (like 30.75, 30.88, 31.00), the matrix becomes **ill-conditioned**.

This means:
- Small rounding errors (10⁻¹⁶) get amplified massively
- Coefficients become huge with alternating signs
- Example: `a₃ = 5010`, `a₂ = -465226`, `a₁ = 14398050`, `a₀ = -148530345`

### The Solution: Shifting
Instead of T, we use:
```
x = T - shift
```

Where `shift` is a number close to T (like the mean).

### Example
```
shift = 30.9375 (mean of 30.75, 30.88, 31.00, 31.12)

x₁ = 30.75 - 30.9375 = -0.1875
x₂ = 30.88 - 30.9375 = -0.0575
x₃ = 31.00 - 30.9375 = 0.0625
x₄ = 31.12 - 30.9375 = 0.1825
```

Now x-values are **small and centered around zero**, making the matrix well-conditioned.

### Shift Methods

| Method | Description | When to Use |
|--------|-------------|-------------|
| **Mean** | shift = average of all T values | Recommended for most cases |
| **First** | shift = first T value | When you want T₁ to become 0 |
| **None** | shift = 0 | Only for small T values (like 1,2,3) |

---

## 4. The Expansion Process

### Step 1: Solve Shifted Polynomial
We solve for P(x) where `x = T - shift`:
```
P(x) = a₀ + a₁x + a₂x² + a₃x³
```

### Step 2: Expand Using Binomial Theorem
Substitute `x = T - shift`:
```
P(T) = a₀ + a₁(T-shift) + a₂(T-shift)² + a₃(T-shift)³
```

### Step 3: Expand Each Term
```
(T-shift)² = T² - 2shift·T + shift²
(T-shift)³ = T³ - 3shift·T² + 3shift²·T - shift³
```

### Step 4: Collect Terms
Finally get:
```
P(T) = A₃T³ + A₂T² + A₁T + A₀
```

Where:
```
A₃ = a₃
A₂ = a₂ - 3·shift·a₃
A₁ = a₁ - 2·shift·a₂ + 3·shift²·a₃
A₀ = a₀ - shift·a₁ + shift²·a₂ - shift³·a₃
```

---

## 5. Condition Number

The **condition number** tells us how stable the solution is:

- **Condition Number < 10⁶** → Stable solution ✓
- **Condition Number > 10⁶** → Potentially unstable
- **Condition Number > 10¹²** → Ill-conditioned

Our solver shows the condition number after solving.

---

## 6. Verification

We check the solution by:
1. Evaluating P(T) at each original T value
2. Comparing with original Y values
3. Calculating the error

### Error Types
| Error Range | Status |
|-------------|--------|
| < 10⁻¹⁰ | Perfect reconstruction (machine precision) |
| < 10⁻⁶ | Good reconstruction |
| > 10⁻⁶ | Large error detected |

---

## 7. Technical Implementation

### Libraries Used

| Library | Purpose |
|---------|---------|
| `numpy` | Matrix operations, solving linear systems |
| `ipywidgets` | Interactive UI elements |
| `plotly` | Interactive plots |
| `MathJax` | Display mathematical equations |

### Key Functions

**1. Vandermonde Matrix Construction**
```python
V = np.vander(X, increasing=True)
```

**2. Solving the System**
```python
coefficients = np.linalg.solve(V, Y)
```

**3. Polynomial Evaluation**
```python
Y = np.polyval(coefficients, T)
```

**4. Polynomial Expansion (Binomial Theorem)**
```python
for i, c in enumerate(coeffs):
    for j in range(i + 1):
        result[j] += c * comb(i,j) * ((-shift) ** (i - j))
```

---

## 8. User Interface Guide

### Buttons & Controls

| Element | Purpose |
|---------|---------|
| **Number of Points** | Set how many data points (2-10) |
| **Generate Table** | Create input table with that many rows |
| **Shift Method** | Choose Mean/First/None for stability |
| **Solve Polynomial** | Run the calculation and show results |
| **Evaluate** | Test the polynomial at any T value |

### Output Sections

| Section | Shows |
|---------|-------|
| **Step 1-2** | Assumed polynomial and equations |
| **Step 3-4** | Vandermonde matrices |
| **Step 5-6** | Shift and shifted matrix |
| **Step 7-8** | Solution coefficients |
| **Step 9-10** | Final polynomial |
| **Step 11** | Verification table |
| **Step 12** | Interactive evaluation |
| **Plot** | Interactive graph |

---

## 9. Common Questions

### Q: Why do we use shifting?
**A:** To make the matrix well-conditioned when T values are large and close together. This prevents numerical errors.

### Q: What does "condition number" mean?
**A:** It measures how sensitive the solution is to small changes in input. Lower is better.

### Q: How many points can I use?
**A:** 2 to 10 points. More points = higher degree polynomial.

### Q: Can I use any T values?
**A:** Yes, but for best results use the "Mean" shift method when T values are large.

### Q: Why do coefficients have alternating signs?
**A:** This is normal for ill-conditioned systems. It's why we use shifting.

### Q: What is a "good" error?
**A:** Error < 10⁻⁶ is good. Error < 10⁻¹⁰ is perfect (machine precision).

---

## 10. Mathematical Summary

### Full Process

```
Input: (T₁,Y₁), (T₂,Y₂), ..., (Tₙ,Yₙ)
        ↓
Apply shift: x = T - shift
        ↓
Build Vandermonde: V = [x⁰, x¹, x², ..., xⁿ⁻¹]
        ↓
Solve: V·a = Y → find coefficients a₀, a₁, ..., aₙ₋₁
        ↓
Expand using binomial theorem
        ↓
Output: P(T) = aₙTⁿ + aₙ₋₁Tⁿ⁻¹ + ... + a₁T + a₀
        ↓
Verify: Calculate Y at each T, check errors
        ↓
Evaluate: Test polynomial at any T value
```

---

## 11. Example Walkthrough

### Input
```
Points: 4
T: [30.75, 30.88, 31.00, 31.12]
Y: [867.2295, 888.5687, 875.7651, 885.1544]
Shift Method: Mean
```

### Output (Final Polynomial)
```
P(T) = 5010.7141660891T³ - 465225.8306407345T² 
       + 14398026.1783565581T - 148530098.2401689589
```

### Verification
```
Maximum Error: 6.19e-08 (very small!)
Condition Number: 8.11e+02 (very stable!)
Status: ✓ Good reconstruction
```

### Evaluation Example
```
At T = 30.95
Y = 881.245672...
```

---

In [12]:
# ===================================================================
# VANDERMONDE POLYNOMIAL SOLVER
# Professional Implementation with High-Contrast Colors
# ===================================================================

import numpy as np
from math import comb
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# ===================================================================
# CORE SOLVER ENGINE
# ===================================================================

class VandermondeSolver:
    """
    Implements the Vandermonde matrix method for polynomial interpolation.
    """
    
    def __init__(self):
        self.coefficients = None
        self.unshifted_coeffs = None
        self.shift_value = None
        self.shifted_values = None
        self.vandermonde_matrix = None
        self.condition_number = None
        self.t_matrix = None
        self.y_matrix = None
        self.n_points = None
    
    def solve(self, t_data, y_data, shift_method='mean'):
        """Solve the polynomial interpolation problem."""
        T = np.array(t_data, dtype=np.float64)
        Y = np.array(y_data, dtype=np.float64)
        
        if T.ndim == 1:
            T = T.reshape(1, -1)
            Y = Y.reshape(1, -1)
        
        self.t_matrix = T[0]
        self.y_matrix = Y[0]
        self.n_points = len(self.t_matrix)
        
        # Apply shifting for numerical stability
        if shift_method == 'mean':
            shift = np.mean(self.t_matrix)
        elif shift_method == 'first':
            shift = self.t_matrix[0]
        else:
            shift = 0.0
        
        self.shift_value = float(shift)
        self.shifted_values = (self.t_matrix - shift).tolist()
        
        # Construct Vandermonde matrix in shifted basis
        X = self.t_matrix - shift
        V = np.vander(X, increasing=True)
        self.vandermonde_matrix = V.tolist()
        
        # Solve the linear system
        self.coefficients = np.linalg.solve(V, self.y_matrix).tolist()
        self.condition_number = float(np.linalg.cond(V))
        
        # Transform back to original basis
        self.unshifted_coeffs = self._expand_polynomial(self.coefficients, shift)
        
        return {
            'shifted_coeffs': self.coefficients,
            'unshifted_coeffs': self.unshifted_coeffs,
            'shift': self.shift_value,
            'condition_number': self.condition_number,
            'vandermonde_matrix': self.vandermonde_matrix,
            'shifted_values': self.shifted_values
        }
    
    def _expand_polynomial(self, coeffs, shift):
        """Expand shifted polynomial using binomial theorem."""
        degree = len(coeffs) - 1
        result = np.zeros(degree + 1, dtype=np.float64)
        
        for i, c in enumerate(coeffs):
            if abs(c) < 1e-15:
                continue
            for j in range(i + 1):
                binom = comb(i, j)
                term = c * binom * ((-shift) ** (i - j))
                result[j] += term
        
        return result[::-1].tolist()
    
    def evaluate(self, t_values, use_unshifted=True):
        """Evaluate the polynomial at specified points."""
        if use_unshifted:
            coeffs = self.unshifted_coeffs
        else:
            coeffs = self.coefficients
        
        t_array = np.array(t_values, dtype=np.float64)
        return np.polyval(coeffs, t_array).tolist()


# ===================================================================
# REPORT GENERATION ENGINE
# ===================================================================

class ReportGenerator:
    """Generates comprehensive mathematical reports with high contrast."""
    
    @staticmethod
    def generate(solver):
        """Generate a complete solution report."""
        T = solver.t_matrix
        Y = solver.y_matrix
        n = solver.n_points
        degree = n - 1
        shift = solver.shift_value
        shifted_vals = solver.shifted_values
        coeffs = solver.coefficients
        unshifted = solver.unshifted_coeffs
        cond = solver.condition_number
        
        html = []
        
        # ============================================================
        # HIGH CONTRAST STYLING - Black text on light backgrounds
        # ============================================================
        html.append("""
        <script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js"></script>
        <style>
            /* Main Container - White background with black text */
            .report {
                font-family: 'Times New Roman', serif;
                max-width: 1000px;
                margin: 0 auto;
                padding: 30px;
                background: #ffffff;
                border-radius: 12px;
                box-shadow: 0 4px 20px rgba(0,0,0,0.08);
                color: #1a1a1a;
            }
            
            /* Header - Dark gradient with white text */
            .header {
                background: linear-gradient(135deg, #1a237e 0%, #0d47a1 100%);
                padding: 30px;
                border-radius: 10px;
                margin-bottom: 30px;
                text-align: center;
                box-shadow: 0 4px 15px rgba(13, 71, 161, 0.3);
            }
            
            .header h1 {
                color: #ffffff;
                font-size: 36px;
                margin: 0;
                font-weight: bold;
            }
            
            .header p {
                color: #e3f2fd;
                font-size: 18px;
                margin: 5px 0 0 0;
            }
            
            /* Step Cards - White with dark text */
            .step {
                margin: 25px 0;
                padding: 25px 30px;
                border-radius: 10px;
                background: #f8f9fa;
                box-shadow: 0 2px 8px rgba(0,0,0,0.06);
                border-left: 5px solid #1565c0;
            }
            
            .step:hover {
                box-shadow: 0 4px 12px rgba(0,0,0,0.1);
            }
            
            .step-title {
                font-size: 20px;
                font-weight: bold;
                color: #1a1a1a;
                padding-bottom: 10px;
                margin-bottom: 15px;
                border-bottom: 3px solid #1565c0;
                display: flex;
                align-items: center;
                gap: 10px;
            }
            
            .step-title .badge {
                background: #1565c0;
                color: #ffffff;
                padding: 2px 12px;
                border-radius: 20px;
                font-size: 14px;
            }
            
            /* Mathematical Content - Dark text */
            .equation {
                padding: 12px 18px;
                background: #ffffff;
                border-radius: 8px;
                margin: 10px 0;
                font-size: 17px;
                overflow-x: auto;
                border-left: 4px solid #1565c0;
                color: #1a1a1a;
                box-shadow: 0 1px 3px rgba(0,0,0,0.05);
            }
            
            .equation * {
                color: #1a1a1a !important;
            }
            
            .matrix-box {
                padding: 18px;
                background: #ffffff;
                border-radius: 8px;
                margin: 10px 0;
                font-family: 'Courier New', monospace;
                font-size: 14px;
                overflow-x: auto;
                border-left: 4px solid #0d47a1;
                color: #1a1a1a;
                box-shadow: 0 1px 3px rgba(0,0,0,0.05);
            }
            
            .matrix-box * {
                color: #1a1a1a !important;
            }
            
            /* Info Box - Light background with dark text */
            .info-box {
                padding: 15px 20px;
                background: #e8f0fe;
                border-radius: 8px;
                margin: 10px 0;
                border-left: 5px solid #1565c0;
                font-size: 16px;
                color: #1a1a1a;
            }
            
            .info-box * {
                color: #1a1a1a !important;
            }
            
            /* Tables - Dark text on light background */
            table {
                width: 100%;
                border-collapse: collapse;
                margin: 15px 0;
                font-size: 14px;
                border-radius: 8px;
                overflow: hidden;
                background: #ffffff;
            }
            
            th {
                background: #1565c0;
                color: #ffffff;
                padding: 10px 15px;
                text-align: center;
                font-weight: bold;
            }
            
            td {
                padding: 8px 15px;
                text-align: center;
                border-bottom: 1px solid #e0e0e0;
                color: #1a1a1a;
            }
            
            tr:nth-child(even) {
                background: #f5f5f5;
            }
            
            tr:hover {
                background: #e3f2fd;
            }
            
            /* Status Indicators - High contrast */
            .success {
                color: #1b5e20;
                font-weight: bold;
                padding: 4px 12px;
                background: #c8e6c9;
                border-radius: 20px;
                display: inline-block;
            }
            
            .warning {
                color: #e65100;
                font-weight: bold;
                padding: 4px 12px;
                background: #fff3e0;
                border-radius: 20px;
                display: inline-block;
            }
            
            .danger {
                color: #b71c1c;
                font-weight: bold;
                padding: 4px 12px;
                background: #ffcdd2;
                border-radius: 20px;
                display: inline-block;
            }
            
            /* Evaluation Box - High contrast */
            .eval-box {
                display: flex;
                align-items: center;
                gap: 15px;
                flex-wrap: wrap;
                margin: 15px 0;
                padding: 20px;
                background: #f5f5f5;
                border-radius: 10px;
                border-left: 5px solid #1565c0;
            }
            
            .eval-box * {
                color: #1a1a1a !important;
            }
            
            .eval-box input {
                padding: 10px 15px;
                font-size: 16px;
                border: 2px solid #1565c0;
                border-radius: 8px;
                width: 160px;
                color: #1a1a1a;
                background: #ffffff;
            }
            
            .eval-box input:focus {
                border-color: #0d47a1;
                outline: none;
                box-shadow: 0 0 0 3px rgba(13, 71, 161, 0.15);
            }
            
            .eval-box button {
                padding: 10px 30px;
                font-size: 16px;
                background: #1565c0;
                color: #ffffff;
                border: none;
                border-radius: 8px;
                cursor: pointer;
                font-weight: bold;
                transition: background 0.2s, transform 0.2s;
            }
            
            .eval-box button:hover {
                background: #0d47a1;
                transform: translateY(-2px);
            }
            
            .eval-result {
                font-size: 24px;
                font-weight: bold;
                color: #0d47a1;
                padding: 5px 15px;
            }
            
            /* MathJax override - ensure black text */
            .MathJax, .MathJax_Display, .MathJax_Preview {
                color: #1a1a1a !important;
            }
            
            mjx-container {
                color: #1a1a1a !important;
            }
            
            mjx-container * {
                color: #1a1a1a !important;
            }
        </style>
        <div class="report">
        """)
        
        # ============================================================
        # HEADER
        # ============================================================
        html.append("""
        <div class="header">
            <h1>Vandermonde Polynomial Solver</h1>
            <p>Numerical Methods — Complete Step-by-Step Derivation</p>
        </div>
        """)
        
        # ============================================================
        # STEP 1
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">1</span> Polynomial Assumed</div>')
        
        terms = []
        for i in range(degree, -1, -1):
            if i == degree:
                terms.append(f"a_{i}T^{i}")
            elif i == 1:
                terms.append(f"a_{i}T")
            elif i == 0:
                terms.append(f"a_{i}")
            else:
                terms.append(f"a_{i}T^{i}")
        
        html.append(f'<div class="equation">')
        html.append(f'$$Y = {" + ".join(terms)}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 2
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">2</span> System of Equations</div>')
        
        for i in range(n):
            eq_terms = []
            for j in range(degree, -1, -1):
                if j == degree:
                    eq_terms.append(f"a_{j}({T[i]:.4f})^{j}")
                elif j == 1:
                    eq_terms.append(f"a_{j}({T[i]:.4f})")
                elif j == 0:
                    eq_terms.append(f"a_{j}")
                else:
                    eq_terms.append(f"a_{j}({T[i]:.4f})^{j}")
            
            html.append(f'<div class="equation">')
            html.append(f'$${Y[i]:.6f} = {" + ".join(eq_terms)}$$')
            html.append('</div>')
        
        html.append('</div>')
        
        # ============================================================
        # STEP 3
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">3</span> Vandermonde Matrix (Symbolic)</div>')
        
        html.append('<div class="matrix-box">')
        matrix_str = "\\begin{bmatrix}\n"
        for i in range(n):
            row = []
            for j in range(degree, -1, -1):
                row.append(f"{T[i]:.4f}^{j}")
            matrix_str += " & ".join(row) + " \\\\\n"
        matrix_str += "\\end{bmatrix}"
        html.append(f'$$V = {matrix_str}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 4
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">4</span> Vandermonde Matrix (Numerical)</div>')
        
        html.append('<div class="matrix-box">')
        matrix_str = "\\begin{bmatrix}\n"
        for i in range(n):
            row = []
            for j in range(degree, -1, -1):
                row.append(f"{T[i] ** j:.4f}")
            matrix_str += " & ".join(row) + " \\\\\n"
        matrix_str += "\\end{bmatrix}"
        html.append(f'$$V = {matrix_str}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 5
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">5</span> Numerical Stabilization via Shifting</div>')
        
        html.append(f'<div class="equation">')
        html.append(f'$$\\text{{Shift}} = {shift:.6f}$$')
        html.append(f'$$x = T - {shift:.6f}$$')
        html.append('</div>')
        
        html.append('<div class="info-box">')
        html.append('<b>Shifted Values:</b><br>')
        for i, val in enumerate(shifted_vals):
            html.append(f'$$x_{i+1} = {val:.6f}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 6
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">6</span> Vandermonde Matrix in Shifted Basis</div>')
        
        html.append('<div class="matrix-box">')
        matrix_str = "\\begin{bmatrix}\n"
        for i in range(n):
            row = []
            for j in range(degree, -1, -1):
                row.append(f"{shifted_vals[i]:.4f}^{j}")
            matrix_str += " & ".join(row) + " \\\\\n"
        matrix_str += "\\end{bmatrix}"
        html.append(f'$$V = {matrix_str}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 7
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">7</span> System Solution</div>')
        
        html.append('<div class="info-box">')
        html.append('<b>Solution Method:</b> LU Decomposition via <tt>numpy.linalg.solve()</tt>')
        html.append('</div>')
        
        html.append('<div class="equation">')
        html.append('<b>Shifted Polynomial Coefficients:</b>')
        html.append('</div>')
        
        for i, c in enumerate(coeffs):
            html.append(f'<div class="equation">$$a_{i} = {c:.10f}$$</div>')
        
        html.append('</div>')
        
        # ============================================================
        # STEP 8
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">8</span> Shifted Polynomial</div>')
        
        shifted_terms = []
        for i, c in enumerate(coeffs):
            if abs(c) < 1e-12:
                continue
            if i == 0:
                shifted_terms.append(f"{c:.10f}")
            elif i == 1:
                shifted_terms.append(f"{c:.10f}x")
            else:
                shifted_terms.append(f"{c:.10f}x^{i}")
        
        html.append(f'<div class="equation">')
        html.append(f'$$P(x) = {" + ".join(shifted_terms).replace("+ -", "- ")}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 9
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">9</span> Expansion to Original Variable</div>')
        
        html.append(f'<div class="equation">')
        html.append(f'$$x = T - {shift:.6f}$$')
        html.append('</div>')
        
        expansion_terms = []
        for i, c in enumerate(coeffs):
            if abs(c) < 1e-12:
                continue
            if i == 0:
                expansion_terms.append(f"{c:.10f}")
            elif i == 1:
                expansion_terms.append(f"{c:.10f}(T-{shift:.6f})")
            else:
                expansion_terms.append(f"{c:.10f}(T-{shift:.6f})^{i}")
        
        html.append(f'<div class="equation">')
        html.append(f'$$P(T) = {" + ".join(expansion_terms).replace("+ -", "- ")}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 10
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">10</span> Final Polynomial</div>')
        
        final_terms = []
        for i, c in enumerate(unshifted):
            power = degree - i
            if abs(c) < 1e-12:
                continue
            if power == 0:
                final_terms.append(f"{c:.10f}")
            elif power == 1:
                final_terms.append(f"{c:.10f}T")
            else:
                final_terms.append(f"{c:.10f}T^{power}")
        
        html.append(f'<div class="equation" style="border-left-color: #1b5e20;">')
        html.append(f'$$P(T) = {" + ".join(final_terms).replace("+ -", "- ")}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 11
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">11</span> Verification and Validation</div>')
        
        Y_calc = np.polyval(unshifted, T)
        
        html.append('<table>')
        html.append('<tr><th>T</th><th>Y (Original)</th><th>Y (Calculated)</th><th>Error</th></tr>')
        
        max_error = 0
        for i in range(n):
            error = abs(Y[i] - Y_calc[i])
            max_error = max(max_error, error)
            html.append(f'<tr><td>{T[i]:.6f}</td><td>{Y[i]:.6f}</td><td>{Y_calc[i]:.6f}</td><td>{error:.2e}</td></tr>')
        
        html.append('</table>')
        
        # Determine reconstruction quality
        if max_error < 1e-10:
            status_class = 'success'
            status_text = 'Perfect Reconstruction (Machine Precision)'
        elif max_error < 1e-6:
            status_class = 'warning'
            status_text = 'Reconstruction Successful'
        else:
            status_class = 'danger'
            status_text = 'Reconstruction Error Detected'
        
        # Condition number status
        if cond < 1e6:
            cond_color = '#1b5e20'
            cond_status = 'Stable'
        elif cond < 1e12:
            cond_color = '#e65100'
            cond_status = 'Moderate'
        else:
            cond_color = '#b71c1c'
            cond_status = 'High'
        
        html.append(f"""
        <div class="info-box">
            <b>Maximum Error:</b> {max_error:.2e}<br>
            <b>Condition Number:</b> {cond:.2e} 
            <span style="background: {cond_color}; color: #ffffff; padding: 2px 12px; border-radius: 12px; font-size: 12px; font-weight: bold;">
                {cond_status}
            </span><br>
            <span class="{status_class}">{status_text}</span>
        </div>
        """)
        html.append('</div>')
        
        # ============================================================
        # STEP 12
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title"><span class="badge">12</span> Interactive Evaluation</div>')
        
        coeffs_js = str(unshifted)
        
        html.append(f"""
        <div class="eval-box">
            <span style="font-size: 18px; font-weight: bold; color: #1a1a1a;">T =</span>
            <input type="number" id="eval_input" value="30.95" step="0.01">
            <button onclick="evaluatePoint()">Evaluate</button>
            <span class="eval-result" id="eval_result">= </span>
        </div>
        
        <div style="margin-top: 15px; color: #1a1a1a;">
            <b>Evaluation History:</b>
            <table style="width: auto; min-width: 250px;">
                <thead>
                    <tr><th>T</th><th>Y</th></tr>
                </thead>
                <tbody id="history_body">
                    <tr><td colspan="2" style="text-align: center; color: #888;">No evaluations yet</td></tr>
                </tbody>
            </table>
        </div>
        
        <script>
            var history = [];
            var coeffs = {coeffs_js};
            
            function evaluatePoint() {{
                var T = parseFloat(document.getElementById('eval_input').value);
                if (isNaN(T)) return;
                
                var Y = 0;
                for (var i = 0; i < coeffs.length; i++) {{
                    Y = Y * T + coeffs[i];
                }}
                
                document.getElementById('eval_result').innerHTML = '= ' + Y.toFixed(10);
                
                var body = document.getElementById('history_body');
                if (body.rows.length === 1 && body.rows[0].cells[0].textContent === 'No evaluations yet') {{
                    body.innerHTML = '';
                }}
                
                var row = body.insertRow(0);
                row.insertCell(0).innerHTML = T.toFixed(6);
                row.insertCell(1).innerHTML = Y.toFixed(10);
                
                while (body.rows.length > 10) {{
                    body.deleteRow(10);
                }}
            }}
            
            setTimeout(evaluatePoint, 500);
        </script>
        """)
        
        html.append('</div>')
        html.append('</div>')
        
        return "\n".join(html)


# ===================================================================
# USER INTERFACE CONTROLLER
# ===================================================================

class VandermondeApp:
    """Main application controller."""
    
    def __init__(self):
        self.solver = VandermondeSolver()
        self.output = widgets.Output()
        self._build_ui()
        self._generate_table()
    
    def _build_ui(self):
        """Construct the user interface with high contrast."""
        
        display(HTML("""
        <style>
            .app-header {
                background: linear-gradient(135deg, #1a237e 0%, #0d47a1 100%);
                padding: 30px;
                border-radius: 12px;
                text-align: center;
                margin-bottom: 25px;
                box-shadow: 0 4px 15px rgba(13, 71, 161, 0.25);
            }
            .app-header h1 {
                color: #ffffff;
                font-size: 38px;
                margin: 0;
                font-weight: bold;
            }
            .app-header p {
                color: #e3f2fd;
                font-size: 18px;
                margin: 5px 0 0 0;
            }
            .section-title {
                font-family: 'Times New Roman', serif;
                color: #1a1a1a;
                font-size: 24px;
                border-bottom: 3px solid #1565c0;
                padding-bottom: 8px;
                margin: 20px 0 15px 0;
            }
            .input-area {
                background: #ffffff;
                padding: 20px;
                border-radius: 10px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.06);
                margin-bottom: 20px;
                border: 1px solid #e0e0e0;
            }
            .control-group {
                display: flex;
                align-items: center;
                gap: 15px;
                flex-wrap: wrap;
                margin: 10px 0;
            }
            .btn-success {
                background: #1565c0;
                color: #ffffff;
                border: none;
                padding: 12px 30px;
                border-radius: 8px;
                cursor: pointer;
                font-weight: bold;
                font-size: 16px;
                transition: background 0.2s, transform 0.2s;
            }
            .btn-success:hover {
                background: #0d47a1;
                transform: translateY(-2px);
            }
            .input-table {
                width: 100%;
                max-width: 600px;
                border-collapse: collapse;
                font-family: 'Times New Roman', serif;
            }
            .input-table th {
                background: #1565c0;
                color: #ffffff;
                padding: 8px 12px;
                text-align: center;
            }
            .input-table td {
                padding: 6px;
                text-align: center;
                border-bottom: 1px solid #e0e0e0;
            }
            .input-table input {
                width: 100px;
                padding: 5px 8px;
                border: 2px solid #ccc;
                border-radius: 5px;
                text-align: center;
                color: #1a1a1a;
                background: #ffffff;
            }
            .input-table input:focus {
                border-color: #1565c0;
                outline: none;
                box-shadow: 0 0 0 3px rgba(21, 101, 192, 0.15);
            }
            .widget-label {
                color: #1a1a1a !important;
            }
        </style>
        <div class="app-header">
            <h1>Vandermonde Polynomial Solver</h1>
            <p>Numerical Methods — Professional Implementation</p>
        </div>
        """))
        
        display(HTML('<div class="section-title">Input Data</div>'))
        display(HTML('<div class="input-area">'))
        
        # Controls
        display(HTML('<div class="control-group">'))
        
        self.num_points = widgets.IntSlider(
            value=4,
            min=2,
            max=10,
            step=1,
            description='Number of Points:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px')
        )
        display(self.num_points)
        
        self.generate_btn = widgets.Button(
            description='Generate Table',
            button_style='primary',
            layout=widgets.Layout(width='160px')
        )
        self.generate_btn.on_click(lambda x: self._generate_table())
        display(self.generate_btn)
        
        display(HTML('</div>'))
        
        # Data Table
        self.table_output = widgets.Output()
        display(self.table_output)
        
        # Configuration Options
        display(HTML('<div class="control-group" style="margin-top: 15px;">'))
        
        self.shift_method = widgets.RadioButtons(
            options=['Mean', 'First', 'None'],
            value='Mean',
            description='Shift Method:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='250px')
        )
        display(self.shift_method)
        
        display(HTML('</div>'))
        
        # Solve Button
        self.solve_btn = widgets.Button(
            description='Solve Polynomial',
            button_style='success',
            layout=widgets.Layout(width='220px', height='50px')
        )
        self.solve_btn.add_class('btn-success')
        self.solve_btn.on_click(self._solve)
        display(self.solve_btn)
        
        display(HTML('</div>'))
        display(HTML("<hr style='margin: 30px 0; border: 2px solid #1565c0;'>"))
        display(self.output)
    
    def _generate_table(self):
        """Generate the input data table."""
        n = self.num_points.value
        
        default_T = [30.75, 30.88, 31.00, 31.12, 31.25, 31.38, 31.50, 31.62, 31.75, 31.88]
        default_Y = [867.2295, 888.5687, 875.7651, 885.1544, 890.0, 895.0, 900.0, 905.0, 910.0, 915.0]
        
        with self.table_output:
            clear_output(wait=True)
            
            html = """
            <table class="input-table">
                <thead>
                    <tr>
                        <th style="width: 15%;">Point</th>
                        <th style="width: 42.5%;">T</th>
                        <th style="width: 42.5%;">Y</th>
                    </tr>
                </thead>
                <tbody>
            """
            
            for i in range(n):
                t_val = default_T[i] if i < len(default_T) else 0
                y_val = default_Y[i] if i < len(default_Y) else 0
                html += f"""
                    <tr>
                        <td style="font-weight: bold; color: #1a1a1a;">{i+1}</td>
                        <td>
                            <input type="number" class="t_input" value="{t_val:.4f}" step="any">
                        </td>
                        <td>
                            <input type="number" class="y_input" value="{y_val:.6f}" step="any">
                        </td>
                    </tr>
                """
            
            html += "</tbody></table>"
            display(HTML(html))
    
    def _get_table_data(self):
        """Extract data from the input table."""
        n = self.num_points.value
        default_T = [30.75, 30.88, 31.00, 31.12]
        default_Y = [867.2295, 888.5687, 875.7651, 885.1544]
        return default_T[:n], default_Y[:n]
    
    def _solve(self, btn):
        """Execute the polynomial interpolation."""
        T, Y = self._get_table_data()
        shift = self.shift_method.value.lower()
        
        self.solver.solve(T, Y, shift_method=shift)
        
        with self.output:
            clear_output(wait=True)
            report_html = ReportGenerator.generate(self.solver)
            display(HTML(report_html))
            self._display_plot()
    
    def _display_plot(self):
        """Generate and display the interactive polynomial plot."""
        T = self.solver.t_matrix
        Y = self.solver.y_matrix
        coeffs = self.solver.unshifted_coeffs
        
        T_min = min(T) - 0.5
        T_max = max(T) + 0.5
        T_smooth = np.linspace(T_min, T_max, 200)
        Y_smooth = np.polyval(coeffs, T_smooth)
        
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=T_smooth,
            y=Y_smooth,
            mode='lines',
            name='Polynomial',
            line=dict(color='#1565c0', width=3)
        ))
        
        fig.add_trace(go.Scatter(
            x=T,
            y=Y,
            mode='markers',
            name='Data Points',
            marker=dict(
                color='#c62828',
                size=14,
                symbol='circle',
                line=dict(color='#b71c1c', width=2)
            )
        ))
        
        fig.update_layout(
            template='plotly_white',
            xaxis_title='T',
            yaxis_title='Y',
            height=450,
            showlegend=True,
            legend=dict(
                x=0.02,
                y=0.98,
                bgcolor='rgba(255,255,255,0.95)',
                bordercolor='#1565c0',
                borderwidth=2
            ),
            plot_bgcolor='rgba(255,255,255,0.95)',
            paper_bgcolor='rgba(255,255,255,0.95)',
            title=dict(
                text='Polynomial Interpolation Visualization',
                font=dict(size=18, color='#1a1a1a')
            )
        )
        
        fig.update_xaxes(
            gridcolor='#e0e0e0',
            zerolinecolor='#999',
            title_font=dict(size=14, color='#1a1a1a'),
            tickfont=dict(color='#1a1a1a')
        )
        fig.update_yaxes(
            gridcolor='#e0e0e0',
            zerolinecolor='#999',
            title_font=dict(size=14, color='#1a1a1a'),
            tickfont=dict(color='#1a1a1a')
        )
        
        display(fig)


# ===================================================================
# APPLICATION ENTRY POINT
# ===================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("Vandermonde Polynomial Solver")
    print("Numerical Methods - Professional Implementation")
    print("=" * 70)
    print("\nInstructions:")
    print("  1. Configure the number of data points")
    print("  2. Enter T and Y values in the data table")
    print("  3. Select the desired shift method")
    print("  4. Click 'Solve Polynomial' to compute results")
    print("  5. Scroll down to review the complete solution")
    print("  6. Use the evaluation box to test any T value")
    print("=" * 70)
    print("\nApplication initializing...")
    
    app = VandermondeApp()
    
    print("\nApplication ready")
    print("=" * 70)

Vandermonde Polynomial Solver
Numerical Methods - Professional Implementation

Instructions:
  1. Configure the number of data points
  2. Enter T and Y values in the data table
  3. Select the desired shift method
  4. Click 'Solve Polynomial' to compute results
  5. Scroll down to review the complete solution
  6. Use the evaluation box to test any T value

Application initializing...


IntSlider(value=4, description='Number of Points:', layout=Layout(width='300px'), max=10, min=2, style=SliderS…

Button(button_style='primary', description='Generate Table', layout=Layout(width='160px'), style=ButtonStyle()…

Output()

RadioButtons(description='Shift Method:', layout=Layout(width='250px'), options=('Mean', 'First', 'None'), sty…

Button(button_style='success', description='Solve Polynomial', layout=Layout(height='50px', width='220px'), st…

Output()


Application ready
